# 010 — Prepare Aerial Data

Downloads aerial tiles intersecting lupine polygons, rasterizes lupine masks,
tiles to 512×512px, and runs QA checks.

**Requires:** `geoai-py fiona` — install via `pip install geoai-py fiona`
Run before `020_train_aerial.ipynb`.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
from config import *
from helpers import *
import random
import geoai
import fiona
from tqdm import tqdm

print(f"BASE_DIR: {BASE_DIR}")
print(f"Cores:    {N_CORES}")

# Check only what 010 needs
for path, label in [
    (GRID_PATH,   "Grid shapefile"),
    (LUPINE_VEC,  "Lupine polygons"),
]:
    print(f"  {'✓' if os.path.exists(path) else '[!] MISSING'}  {label}")

In [ ]:
# ── Step 1: load lupine polygons and grid ─────────────────────────────────────
print("── Step 1: Load lupine and grid ────────────────────────────")

lupine = gpd.read_file(LUPINE_VEC, layer=LUPINE_LAYER)
lupine = lupine.to_crs("EPSG:8088")
lupine["geometry"] = lupine.geometry.make_valid()
print(f"Lupine polygons: {len(lupine)}  CRS: {lupine.crs}")

grid = gpd.read_file(GRID_PATH).to_crs(lupine.crs)
print(f"Grid cells: {len(grid)}")

# Find tiles intersecting lupine — deduplicate on tile label
intersecting_tiles = gpd.sjoin(
    grid, lupine[["geometry"]],
    predicate="intersects", how="inner"
).drop_duplicates(subset=["label"])

urls    = intersecting_tiles["path"].unique().tolist()
url_map = {os.path.basename(url): url for url in urls}

print(f"Intersecting tiles: {len(intersecting_tiles)}")
print(f"Download URLs:      {len(urls)}")

In [ ]:
# ── Step 2: download tiles ────────────────────────────────────────────────────
print("── Step 2: Download tiles ──────────────────────────────────")

expected_files   = {os.path.basename(url) for url in urls}
downloaded_files = {f for f in os.listdir(DOWNLOADED_DIR) if f.endswith(".tif")}
to_download      = expected_files - downloaded_files

print(f"Expected:    {len(expected_files)}")
print(f"Downloaded:  {len(downloaded_files)}")
print(f"To download: {len(to_download)}")

for fname in tqdm(sorted(to_download), desc="Downloading"):
    ok = safe_download(url_map[fname], os.path.join(DOWNLOADED_DIR, fname))
    if not ok:
        print(f"  [!] Failed: {fname}")

downloaded_after = {f for f in os.listdir(DOWNLOADED_DIR) if f.endswith(".tif")}
missing          = expected_files - downloaded_after
print(f"\nDownloaded: {len(downloaded_after)}  Missing: {len(missing)}")
if missing:
    print("Missing:", sorted(missing)[:10])

In [ ]:
# ── Step 3: rasterize lupine masks ────────────────────────────────────────────
print("── Step 3: Rasterize lupine masks ──────────────────────────")

raster_files          = sorted(f for f in os.listdir(DOWNLOADED_DIR) if f.endswith(".tif"))
success, skipped, failed = 0, 0, 0

for file in tqdm(raster_files, desc="Creating masks"):
    image_path = os.path.join(DOWNLOADED_DIR, file)
    mask_path  = os.path.join(MASKS_DIR, file.replace(".tif", "_lupine.tif"))

    if os.path.exists(mask_path):
        skipped += 1
        continue
    try:
        geoai.vector_to_raster(lupine, mask_path, reference_raster=image_path)
        success += 1
    except Exception as e:
        print(f"  [!] Failed {file}: {e}")
        failed += 1

print(f"\nCreated: {success}  Skipped: {skipped}  Failed: {failed}")

In [ ]:
# ── Step 4: QA — check image/mask alignment ───────────────────────────────────
print("── Step 4: Alignment QA ────────────────────────────────────")

images = sorted(f for f in os.listdir(DOWNLOADED_DIR) if f.endswith(".tif"))
issues = []

for file in tqdm(images, desc="Checking alignment"):
    img_path  = os.path.join(DOWNLOADED_DIR, file)
    mask_path = os.path.join(MASKS_DIR, file.replace(".tif", "_lupine.tif"))

    if not os.path.exists(mask_path):
        issues.append((file, "missing mask")); continue
    try:
        with rasterio.open(img_path) as img, rasterio.open(mask_path) as msk:
            if (img.width != msk.width) or (img.height != msk.height):
                issues.append((file, "shape mismatch"))
            elif img.crs != msk.crs:
                issues.append((file, "CRS mismatch"))
            elif img.transform != msk.transform:
                issues.append((file, "transform mismatch"))
    except Exception as e:
        issues.append((file, f"error: {e}"))

print(f"Total images: {len(images)}  Issues: {len(issues)}")
for i in issues[:10]:
    print(f"  {i}")

In [ ]:
# ── Step 5: tile images and masks to 512×512px ────────────────────────────────
import shutil

TEMP_IMG  = os.path.join(BASE_DIR, "temp", "img")
TEMP_MASK = os.path.join(BASE_DIR, "temp", "mask")
os.makedirs(TEMP_IMG,  exist_ok=True)
os.makedirs(TEMP_MASK, exist_ok=True)

all_sources    = sorted(f for f in os.listdir(DOWNLOADED_DIR) if f.endswith((".tif", ".tiff")))
existing_tiles = set(os.listdir(TILES_IMG_DIR))
already_done   = {f for f in all_sources
                  if any(t.startswith(os.path.splitext(f)[0]) for t in existing_tiles)}
remaining      = [f for f in all_sources if f not in already_done]

print(f"Source images:  {len(all_sources)}")
print(f"Already tiled:  {len(already_done)}")
print(f"Remaining:      {len(remaining)}")

for i, f in enumerate(remaining):
    stem      = os.path.splitext(f)[0]
    mask_name = f.replace(".tif", "_lupine.tif")
    img_src   = os.path.join(DOWNLOADED_DIR, f)
    mask_src  = os.path.join(MASKS_DIR, mask_name)

    # Clear temp folders
    for tmp in os.listdir(TEMP_IMG):  os.remove(os.path.join(TEMP_IMG, tmp))
    for tmp in os.listdir(TEMP_MASK): os.remove(os.path.join(TEMP_MASK, tmp))

    if not os.path.exists(img_src):
        print(f"  [!] Image missing: {f}")
        continue
    if not os.path.exists(mask_src):
        print(f"  [!] Mask missing: {mask_name}")
        continue

    shutil.copy2(img_src,  os.path.join(TEMP_IMG, f))
    shutil.copy2(mask_src, os.path.join(TEMP_MASK, f))

    try:
        geoai.export_geotiff_tiles_batch(
            images_folder=TEMP_IMG,
            masks_folder=TEMP_MASK,
            output_folder=TILES_DIR,
            tile_size=512,
            stride=512,
            skip_empty_tiles=True,
            quiet=True,
        )
        tiles_now      = set(os.listdir(TILES_IMG_DIR))
        new_count      = len(tiles_now) - len(existing_tiles)
        existing_tiles = tiles_now
        print(f"  [{i+1}/{len(remaining)}] {stem} → {new_count} new tiles")
    except Exception as e:
        print(f"  [!] Error at {f}: {e}")

print(f"\nDone. Total tiles: {len(os.listdir(TILES_IMG_DIR))}")

In [ ]:
# ── Step 6: tile count QA ─────────────────────────────────────────────────────
print("── Step 6: Tile count QA ───────────────────────────────────")

image_set = {f for f in os.listdir(TILES_IMG_DIR)  if f.endswith(".tif")}
mask_set  = {f for f in os.listdir(TILES_MASK_DIR) if f.endswith(".tif")}

print(f"Images: {len(image_set)}  Masks: {len(mask_set)}")
print(f"Missing masks:  {len(image_set - mask_set)}")
print(f"Missing images: {len(mask_set  - image_set)}")

In [ ]:
# ── Step 7: mask coverage QA ──────────────────────────────────────────────────
print("── Step 7: Mask coverage QA ────────────────────────────────")

import pandas as pd
import matplotlib.pyplot as plt

mask_files = [f for f in os.listdir(TILES_MASK_DIR) if f.endswith(".tif")]
results    = []

for f in tqdm(mask_files, desc="Mask QA"):
    path = os.path.join(TILES_MASK_DIR, f)
    try:
        with rasterio.open(path) as src:
            mask   = src.read(1)
            unique = np.unique(mask)
            results.append({
                "file":     f,
                "coverage": float(np.mean(mask > 0)),
                "binary":   set(unique).issubset({0, 1}),
                "empty":    np.all(mask == 0),
            })
    except Exception as e:
        results.append({"file": f, "error": str(e)})

df    = pd.DataFrame(results)
valid = df[~df["coverage"].isna()]

print(f"Total masks:  {len(df)}")
print(f"Binary masks: {valid['binary'].sum()}")
print(f"Empty masks:  {valid['empty'].sum()}")
print(f"Coverage — mean: {valid['coverage'].mean():.4f}  median: {valid['coverage'].median():.4f}")

df.to_csv(MASK_QA_PATH, index=False)
print(f"QA saved → {MASK_QA_PATH}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Lupine Coverage — 512×512px tiles", fontsize=13)
axes[0].hist(valid["coverage"], bins=50)
axes[0].axvline(valid["coverage"].mean(),   color="red",   linestyle="--",
                label=f"mean {valid['coverage'].mean():.3f}")
axes[0].axvline(valid["coverage"].median(), color="green", linestyle="--",
                label=f"median {valid['coverage'].median():.3f}")
axes[0].set_xlabel("Coverage"); axes[0].legend()
counts = np.histogram(valid["coverage"], bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0])[0]
axes[1].bar(["0-20%","20-40%","40-60%","60-80%","80-100%"], counts)
axes[1].set_xlabel("Coverage range")
plt.tight_layout()
plt.savefig(os.path.join(QA_VISUALS_DIR, "512px_coverage_distribution.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Step 8: visual QA — random tile pairs ─────────────────────────────────────
print("── Step 8: Visual QA ───────────────────────────────────────")

import matplotlib.pyplot as plt

images  = sorted(f for f in os.listdir(TILES_IMG_DIR) if f.endswith(".tif"))
samples = random.sample(images, min(5, len(images)))

for f in samples:
    img_path  = os.path.join(TILES_IMG_DIR, f)
    mask_path = os.path.join(TILES_MASK_DIR, f)
    if not os.path.exists(mask_path):
        continue

    with rasterio.open(img_path) as src:
        img = src.read([1, 2, 3]).transpose(1, 2, 0)
    with rasterio.open(mask_path) as src:
        mask = src.read(1)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img / 255.0); axes[0].set_title("Image"); axes[0].axis("off")
    axes[1].imshow(mask, cmap="Reds")
    axes[1].set_title(f"Mask ({mask.mean():.2%})"); axes[1].axis("off")
    fig.suptitle(f)
    plt.tight_layout()
    plt.savefig(os.path.join(QA_VISUALS_DIR, f.replace(".tif", "_qa.png")),
                dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ── Step 9: create 500-tile training subset ───────────────────────────────────
print("── Step 9: Training subset ─────────────────────────────────")

import pandas as pd

df_qa  = pd.read_csv(MASK_QA_PATH)
subset = df_qa.sample(500, random_state=99)

print(f"Subset size: {len(subset)}")
print(f"Coverage — mean: {subset['coverage'].mean():.3f}  "
      f"median: {subset['coverage'].median():.3f}")

subset[["file", "coverage"]].to_csv(SUBSET_OUT, index=False)
print(f"Saved → {SUBSET_OUT}")

## Step 10: Spectral profile (optional)

Run this cell to inspect the spectral distribution of lupine-labeled pixels.
Useful after adding new tiles or retraining — not needed every run.

In [ ]:
# ── Step 10: spectral profile of lupine pixels (optional) ─────────────────────
import pandas as pd
import matplotlib.pyplot as plt

all_tiles    = sorted(f for f in os.listdir(TILES_IMG_DIR) if f.endswith(".tif"))
sample_tiles = random.Random(42).sample(all_tiles, min(500, len(all_tiles)))

R_vals, G_vals, B_vals, NIR_vals = [], [], [], []
n_with_lupine = 0

for f in tqdm(sample_tiles, desc="Spectral profile"):
    img_path  = os.path.join(TILES_IMG_DIR, f)
    mask_path = os.path.join(TILES_MASK_DIR, f)
    if not os.path.exists(img_path) or not os.path.exists(mask_path):
        continue

    with rasterio.open(img_path)  as src: img  = src.read().astype(np.float32)
    with rasterio.open(mask_path) as src: mask = src.read(1).astype(np.uint8)

    pos = np.where(mask.reshape(-1) == 1)[0]
    if len(pos) == 0: continue
    if len(pos) > 1000:
        pos = np.random.default_rng(seed=abs(hash(f)) % (2**32)).choice(pos, 1000, replace=False)

    R_vals.append(img[0].reshape(-1)[pos])
    G_vals.append(img[1].reshape(-1)[pos])
    B_vals.append(img[2].reshape(-1)[pos])
    NIR_vals.append(img[3].reshape(-1)[pos])
    n_with_lupine += 1

R_all, G_all = np.concatenate(R_vals), np.concatenate(G_vals)
B_all, NIR_all = np.concatenate(B_vals), np.concatenate(NIR_vals)

eps      = 1e-6
R_n, G_n, B_n, NIR_n = R_all/255, G_all/255, B_all/255, NIR_all/255
NDVI_all = (NIR_n - R_n)   / (NIR_n + R_n   + eps)
NDWI_all = (G_n   - NIR_n) / (G_n   + NIR_n + eps)
denom   = NIR_n + 6*R_n - 7.5*B_n + 1
EVI_all = np.clip(2.5 * (NIR_n - R_n) / np.maximum(np.abs(denom), 0.1), -1, 2)

print(f"Tiles with lupine: {n_with_lupine}  Total pixels: {len(R_all):,}")
print(f"NDVI — mean={NDVI_all.mean():.3f}  "
      f"p5={np.percentile(NDVI_all,5):.3f}  "
      f"p95={np.percentile(NDVI_all,95):.3f}")
print(f"  → Update NDVI_MIN/NDVI_MAX in config.py if needed")

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle(f"Spectral profile — lupine pixels ({len(R_all):,})", fontsize=13)
for ax, (arr, name, color) in zip(axes.flat, [
    (R_all,"R","red"),(G_all,"G","green"),(B_all,"B","blue"),(NIR_all,"NIR","darkred"),
    (NDVI_all,"NDVI","forestgreen"),(NDWI_all,"NDWI","steelblue"),(EVI_all,"EVI","olive")]):
    ax.hist(arr, bins=100, color=color, alpha=0.7)
    ax.axvline(arr.mean(), color="black", linestyle="--", linewidth=1.5,
               label=f"mean {arr.mean():.3f}")
    ax.set_title(name); ax.legend(fontsize=7)
axes[1,3].scatter(NDVI_all[::10], NIR_all[::10], alpha=0.05, s=1, color="green")
axes[1,3].set_title("NIR vs NDVI")
axes[1,3].set_xlabel("NDVI"); axes[1,3].set_ylabel("NIR")
plt.tight_layout()
plt.savefig(os.path.join(QA_VISUALS_DIR, "spectral_profile_lupine.png"),
            dpi=150, bbox_inches="tight")
plt.show()

# Save CSV
summary = []
for name, arr in [("R",R_all),("G",G_all),("B",B_all),("NIR",NIR_all),
                  ("NDVI",NDVI_all),("NDWI",NDWI_all),("EVI",EVI_all)]:
    summary.append({"band": name, "mean": round(float(arr.mean()),4),
                    "p5": round(float(np.percentile(arr,5)),4),
                    "median": round(float(np.median(arr)),4),
                    "p95": round(float(np.percentile(arr,95)),4)})
pd.DataFrame(summary).to_csv(
    os.path.join(DATA_DIR, "spectral_profile_lupine.csv"), index=False)
print("Spectral profile CSV saved ✓")